# Natural Language to SQL with CodeT5+
This notebook implements a text-to-SQL model using Salesforce's `codet5p-770m`, trained on `b-mc2/sql-create-context` and evaluated on the `Spider` validation set.

In [ ]:
!pip install -q transformers datasets accelerate tensorboard

In [ ]:
!pip install -q sentencepiece protobuf evaluate

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import torch


In [ ]:
# 1. Load Datasets
# 'sql-create-context' is excellent for teaching the model DDL-to-SQL mapping
train_dataset = load_dataset('b-mc2/sql-create-context', split='train')

# 'spider' is the academic gold standard for zero-shot cross-domain evaluation
spider_val = load_dataset('spider', split='validation')

print(f'Train set size: {len(train_dataset)}')
print(f'Spider Validation set size: {len(spider_val)}')


# Load Model

In [ ]:
from google.colab import userdata

# Get token
try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = None

# Load Model and Tokenizer
model_id = 'Salesforce/codet5p-770m'

# Define the single device you want to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    use_fast=False, 
    extra_special_tokens=[] 
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, 
    # REMOVE: device_map='auto'
    # ADD: trust_remote_code and tie_word_embeddings for safety
    trust_remote_code=True,
    tie_word_embeddings=False 
)

# Explicitly move the model to your chosen device
model.to(device)

print(f'Model loaded and successfully pinned to: {device}')

In [ ]:
# Preview the training data format
display(train_dataset.to_pandas().head())

### 3. Evaluation Metrics & Visualizations
We will define metrics to track during training and visualize our dataset characteristics.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Visualize query length distribution
train_df = train_dataset.to_pandas()
train_df['query_length'] = train_df['answer'].apply(len)

plt.figure(figsize=(10, 6))
sns.histplot(train_df['query_length'], bins=50, kde=True, color='skyblue')
plt.title('Distribution of SQL Query Lengths (Train Set)')
plt.xlabel('Length of SQL Query')
plt.ylabel('Frequency')
plt.show()

In [ ]:
import numpy as np
import evaluate # The modern replacement for datasets.load_metric

# 1. Load the Exact Match metric from the new library
# In a research context, you might also load "rouge" or "bleu",
# but for SQL, Exact Match is the standard starting point.
em_metric = evaluate.load("exact_match")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # If the model returns logits, we take the argmax to get token IDs
    if isinstance(preds, tuple):
        preds = preds[0]

    # Convert token IDs back to strings
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 (ignore index) in labels so the tokenizer can decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 2. Clean whitespace for a fair comparison
    # Group Project Tip: Students often forget that "SELECT *" and "SELECT  *"
    # are technically different strings but the same SQL. .strip() helps.
    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # 3. Compute using the evaluate library
    results = em_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"exact_match": results["exact_match"]}

print("Metric function defined using 'evaluate' library successfully.")

### 4. PyTorch Dataset and Training Setup
We will define custom Dataset classes to handle tokenization and mapping of the text-to-SQL pairs.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class SQLDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Input: "question: {q} context: {c}"
        input_text = f"question: {item['question']} context: {item['context']}"
        target_text = item['answer']

        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')

        labels = targets.input_ids
        labels[labels == self.tokenizer.pad_token_id] = -100 # Mask padding for loss

        return {
            'input_ids': inputs.input_ids.squeeze(),
            'attention_mask': inputs.attention_mask.squeeze(),
            'labels': labels.squeeze()
        }

# Initialize PyTorch Datasets
train_pt_dataset = SQLDataset(train_dataset, tokenizer)
train_loader = DataLoader(train_pt_dataset, batch_size=8, shuffle=True)

Dataset class for the spider test dataset



In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader

# 1. Load the Spider validation set that is already formatted with 'context'
# This matches the schema-aware training you did with sql-create-context
spider_test_raw = load_dataset("richardr1126/spider-context-validation", split="validation")

# 2. Wrap it in your existing SQLDataset class
# (Make sure your class handles 'question', 'context', and 'query'/'answer' columns)
class SpiderTestDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # Spider-context uses 'query' instead of 'answer', so we check for both
        target_text = item.get('answer') or item.get('query')
        input_text = f"question: {item['question']} context: {item['context']}"

        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')

        labels = targets.input_ids
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': inputs.input_ids.squeeze(),
            'attention_mask': inputs.attention_mask.squeeze(),
            'labels': labels.squeeze()
        }

spider_test_dataset = SpiderTestDataset(spider_test_raw, tokenizer)
spider_test_loader = DataLoader(spider_test_dataset, batch_size=8, shuffle=False)

print(f"Spider Validation Set ready: {len(spider_test_dataset)} samples.")

In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm

# Standard PyTorch Training Loop Setup
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader, desc='Training'):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)

print("PyTorch training environment ready.")

# Execute Training and Spider Evaluation

Run the cells below to start the fine-tuning. It includes a basic "Early Saving" logic to keep your best-performing version.

In [ ]:
from torch.utils.data import Subset
import numpy as np

# Select 10,000 random indices from your 78k training set
subset_indices = np.random.choice(len(train_pt_dataset), 20000, replace=False)
subset_train_dataset = Subset(train_pt_dataset, subset_indices)

# New loader with the subset (keeping batch_size=2 for VRAM safety)
train_loader = DataLoader(subset_train_dataset, batch_size=2, shuffle=True)

print(f"Dataset capped at 20,000 samples.")

In [ ]:
import torch
import os
import gc
import warnings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Adafactor, logging
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# 1. KILL BACKGROUND ERRORS & HUB SYNC
os.environ["TRANSFORMERS_NO_AD_HOC_KHF_TRANSFORMS"] = "1"
# logging.set_verbosity_error() silences the "ReadTimeout" and "AttributeError" spam
logging.set_verbosity_error() 
warnings.filterwarnings("ignore")

# 2. EMERGENCY CLEANUP
if 'model' in globals(): del model
if 'optimizer' in globals(): del optimizer
gc.collect()
torch.cuda.empty_cache()

# 3. SETUP & LOAD
device = torch.device('cuda:0')
model_id = 'Salesforce/codet5p-770m'

print(f"Loading {model_id} (Quiet Mode)...")

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, extra_special_tokens=[])
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_id, 
    torch_dtype=torch.float16, 
    tie_word_embeddings=False,
    trust_remote_code=True
).to(device)

# 4. OPTIMIZATIONS
model.config.use_cache = False
model.gradient_checkpointing_enable()

# 5. ADAFACTOR OPTIMIZER (The VRAM Saver)
optimizer = Adafactor(
    model.parameters(),
    lr=5e-5,
    scale_parameter=False,
    relative_step=False,
    warmup_init=False
)

# 6. DATA & LOOP SETUP (Capped at 20k rows)
train_loader = DataLoader(train_pt_dataset, batch_size=2, shuffle=True)
spider_test_loader = DataLoader(spider_test_dataset, batch_size=2, shuffle=False)
accumulation_steps = 8 
MAX_STEPS = 10000 

print(f"Starting stabilized training on {device}...")

for epoch in range(1):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    train_bar = tqdm(train_loader, desc=f"Training")
    for i, batch in enumerate(train_bar):
        if i >= MAX_STEPS: break
            
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / accumulation_steps
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * accumulation_steps
        train_bar.set_postfix(loss=loss.item() * accumulation_steps)

    # --- EVALUATION ---
#     model.eval()
#     all_preds, all_labels = [], []
#     with torch.no_grad():
#         for batch in tqdm(spider_test_loader, desc=f"Spider Eval"):
#             input_ids = batch['input_ids'].to(device)
#             attention_mask = batch['attention_mask'].to(device)
#             labels = batch['labels']
#             outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=64)
            
#             all_preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
#             clean_labels = labels.clone()
#             clean_labels[clean_labels == -100] = tokenizer.pad_token_id
#             all_labels.extend(tokenizer.batch_decode(clean_labels, skip_special_tokens=True))

#     correct = sum([1 if p.strip().lower() == l.strip().lower() else 0 for p, l in zip(all_preds, all_labels)])
#     print(f"Final Spider Exact Match: {(correct / len(all_preds)) * 100:.2f}%")
    
#     model.save_pretrained(f"results/{model_id.split('/')[-1]}_final")

# print("\nAll background noise silenced. Training complete!")

In [ ]:
print(spider_test_raw[0].keys())

# Model Evaluation

In [ ]:
import torch
from tqdm.auto import tqdm

# We use the 'model' and 'tokenizer' currently in your Kaggle memory
model.eval()
all_preds, all_labels = [], []

print("Starting Spider Evaluation using 'db_info' as context...")

with torch.no_grad():
    # We use a smaller loop or the full set; let's do the full spider_test_raw
    for i in tqdm(range(len(spider_test_raw)), desc="Spider Eval"):
        item = spider_test_raw[i]
        
        # 1. MAP THE KEYS
        question = item['question']
        # Extract schema from db_info (ensure it's a string)
        context = str(item['db_info']) 
        target_sql = item['ground_truth']
        
        # 2. FORMAT PROMPT (Must match your training: "question: {q} context: {c}")
        input_text = f"question: {question} context: {context}"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

        # 3. GENERATE SQL
        # num_beams=2 is a good balance of speed and accuracy
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=128,
            num_beams=2,
            early_stopping=True
        )

        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Clean up for comparison
        all_preds.append(pred_sql.strip().lower())
        all_labels.append(target_sql.strip().lower())

# 4. CALCULATE EXACT MATCH (EM)
# Simple string comparison as a baseline
correct = sum([1 if p == l else 0 for p, l in zip(all_preds, all_labels)])
em_score = (correct / len(all_preds)) * 100

print(f"\n--- EVALUATION RESULTS ---")
print(f"Spider Exact Match Accuracy: {em_score:.2f}%")

# 5. SAVE THE MODEL (Crucial step that was missed due to the crash)
# This saves the fine-tuned weights to your Kaggle output directory
save_path = f"results/{model_id.split('/')[-1]}_final_fixed"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Success! Model weights and tokenizer saved to: {save_path}")

In [ ]:
print("--- DEBUGGING PREDICTIONS ---")
for i in range(5):
    print(f"\nSample {i+1}:")
    print(f"Question: {spider_test_raw[i]['question']}")
    print(f"Model Prediction: '{all_preds[i]}'")
    print(f"Ground Truth:     '{all_labels[i]}'")

In [ ]:
import shutil
import os

# Define the folder you want to download
folder_to_zip = '/kaggle/working/results'
# Define the name of the output zip file
output_filename = 'codet5p_spider_model'

# Create the zip file
shutil.make_archive(output_filename, 'zip', folder_to_zip)

print(f"Archive created: {output_filename}.zip")
print(f"Files inside: {os.listdir(folder_to_zip)}")

In [ ]:
import ast
import json

def generate_ddl_from_spider(db_info):
    # 1. Parse the string if it's not already a dictionary
    if isinstance(db_info, str):
        try:
            # Try parsing as JSON first, then as a Python literal
            db_info = json.loads(db_info.replace("'", '"')) 
        except:
            try:
                db_info = ast.literal_eval(db_info)
            except:
                # If it's already a DDL string, just return it
                return db_info

    # 2. Extract schema parts
    ddl_parts = []
    # Using .get() to avoid further KeyErrors
    table_names = db_info.get('table_names_original', db_info.get('table_names', []))
    column_names = db_info.get('column_names_original', db_info.get('column_names', []))
    column_types = db_info.get('column_types', [])
    
    if not table_names:
        return str(db_info) # Fallback if structure is unknown

    # 3. Generate DDL
    for i, table in enumerate(table_names):
        cols = []
        for col_idx, (t_idx, col_name) in enumerate(column_names):
            if t_idx == i:
                # Get type or default to TEXT
                c_type = column_types[col_idx] if col_idx < len(column_types) else "TEXT"
                cols.append(f"{col_name} {c_type}")
        
        ddl = f"CREATE TABLE {table} ({', '.join(cols)})"
        ddl_parts.append(ddl)
        
    return " ".join(ddl_parts)

In [ ]:
import torch
import re
import json
import ast
from tqdm.auto import tqdm

# 1. SETUP SUBSET
subset_size = 10 # Let's start with 10 to be ultra-fast
model.eval()
subset_preds, subset_labels = [], []

print(f"Running Sanity Check on {subset_size} samples...")

with torch.no_grad():
    for i in range(subset_size):
        # Accessing by index directly is safer than 'for item in test_subset'
        item = spider_test_raw[i]
        
        # SAFETY: If 'item' is a string, turn it into a dict
        if isinstance(item, str):
            try:
                item = json.loads(item.replace("'", '"'))
            except:
                item = ast.literal_eval(item)

        # 2. EXTRACT DATA
        try:
            question = item['question']
            # Using your new DDL generator
            context = generate_ddl_from_spider(item['db_info'])
            target_sql = item['ground_truth']
        except KeyError as e:
            print(f"Skipping index {i} due to missing key: {e}")
            continue

        # 3. INFERENCE
        input_text = f"question: {question} context: {context}"
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True)
        subset_preds.append(pred_sql.strip())
        subset_labels.append(target_sql.strip())

# 4. MINI-EVALUATION
print("\n" + "="*50)
print("SANITY CHECK PREDICTIONS")
print("="*50)

matches = 0
for i in range(len(subset_preds)):
    p = normalize_sql(subset_preds[i])
    l = normalize_sql(subset_labels[i])
    status = "✅" if p == l else "❌"
    if p == l: matches += 1
    
    print(f"\n{status} Q: {spider_test_raw[i]['question']}")
    print(f"   Pred: {subset_preds[i]}")
    print(f"   True: {subset_labels[i]}")

print(f"\nSubset Accuracy: {(matches/len(subset_preds))*100:.2f}%")

The "Oracle" Test (Let's prove it works)

To prove the model is actually smart, let's run a test where we only give it the correct table. If this works, it confirms the problem is "Schema Linking" (picking the right table), not the model's ability to write SQL.

In [ ]:
def get_oracle_schema(db_id, question):
    # For the 'concert_singer' database (first 10 samples), the only relevant table is 'singer'
    if "singer" in question.lower() and "concert" not in question.lower():
        return "CREATE TABLE singer (Singer_ID INT, Name TEXT, Country TEXT, Age INT)"
    return generate_ddl_from_spider(db_id) # Fallback

# Run a quick 5-sample check
for i in range(5):
    q = spider_test_raw[i]['question']
    context = "CREATE TABLE singer (Singer_ID INT, Name TEXT, Country TEXT, Age INT, Song_Name TEXT, Song_release_year TEXT)"
    input_text = f"question: {q} context: {context}"
    
    inputs = tokenizer(input_text, return_tensors="pt").to(device)
    outputs = model.generate(inputs.input_ids, max_length=128)
    print(f"Q: {q}\nModel: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")

This Oracle test is the most important diagnostic we’ve run so far. It reveals that the model isn't "dumb"—it’s actually over-fitted to a pattern in your training data.

The 0% EM score isn't because the model doesn't know SQL; it's because it has developed a Spurious Correlation. It thinks that every column you provide in the CREATE TABLE context must be used as a filter in a WHERE clause. This is why it’s hallucinating things like WHERE Name = "name" AND Country = "country". It’s treating the schema like a "fill-in-the-blanks" form.

# 1. The "Instructional Prompt" Fix
Since this is a 770M model, it relies heavily on patterns. You can "re-calibrate" its behavior by adding a clear instruction and a few "Negative Examples" (where most columns are ignored).

In [ ]:
instruction = "Task: Convert the question into a valid SQL query using ONLY the necessary columns from the context. Do not add filters (WHERE clauses) for columns not mentioned in the question."

input_text = f"{instruction}\n\nQuestion: {question}\nContext: {context}\nSQL:"

inputs = tokenizer(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs.input_ids, max_length=128)
print(f"Q: {q}\nModel: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")

# 2. The "Few-Shot" Strategy

If instructions aren't enough, let's try to give the model 2-3 "Few-Shot" examples in the prompt. This "reminds" the model that it’s okay to ignore columns.

In [ ]:
few_shot_examples = """
Question: How many students are there?
Context: CREATE TABLE students (id INT, name TEXT, age INT, grade TEXT)
SQL: SELECT COUNT(*) FROM students

Question: Show names of all teachers from India.
Context: CREATE TABLE teachers (id INT, name TEXT, country TEXT, subject TEXT)
SQL: SELECT name FROM teachers WHERE country = "India"
"""

input_text = f"{few_shot_examples}\n\nQuestion: {question}\nContext: {context}\nSQL:"

inputs = tokenizer(input_text, return_tensors="pt").to(device)
outputs = model.generate(inputs.input_ids, max_length=128)
print(f"Q: {q}\nModel: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")

At 770M parameters, CodeT5+ doesn't have a large enough "reasoning" engine to truly follow instructions or few-shot examples when they conflict with what it learned during 10,000 steps of fine-tuning.

It has become a Template Matcher.

* In Strategy 1, it saw a complex schema and "hallucinated" a join because it thought a complex schema must require a complex query.

* In Strategy 2, it literally copied the word "India" from my few-shot example. It’s not "reading" the prompt; it’s using it as a copy-paste template.

# Phase 2 Plan

We already likely have the "SQL Logic" burned into the weights. Now ywe need to teach it Schema Linking (picking only what matters). We need to fine-tune on the actual Spider Training Set.

1. Why Spider is different: The sql-create-context dataset we used is "Clean" (usually 1 table). Spider is "Messy" (many tables). Training on Spider teaches the model to ignore 80% of the columns.


# The Training Setup 

A "Correction Pass" of 1 epoch on Spider (about 7,000 samples) with a very low learning rate might fix the "hallucination" habit.

In [ ]:
import torch
import gc
import json
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Adafactor
from tqdm.auto import tqdm

# 1. SETUP & CLEANUP
gc.collect()
torch.cuda.empty_cache()
device = torch.device('cuda:0')

# 2. LOAD OFFICIAL SPIDER TRAIN SPLIT
print("Loading official Spider training data...")
# The official 'spider' dataset has 'train' and 'validation'
spider_train_raw = load_dataset("spider", split="train")

# 3. THE SCHEMA MAPPER
# Spider's DDL info is in a separate 'tables.json'. We'll fetch it to build our DDLs.
# If you are using the HF dataset, the schema is often nested.
def get_ddl_from_hf_item(item):
    """
    Constructs DDL from the official HF Spider format.
    """
    db_id = item['db_id']
    # This is a placeholder logic—most HF Spider datasets require joining 
    # with a tables.json. To keep it simple for your loop:
    return generate_ddl_from_spider(item.get('db_info', ''))

# 4. UPDATED DATASET CLASS
class SpiderTrainDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Mapping official Spider keys
        question = item['question']
        target_sql = item['query']
        
        # We use the DDL generator you added earlier
        # Note: If item['db_info'] is missing, we use a fallback
        context = generate_ddl_from_spider(item.get('db_info', ''))
        
        input_text = f"question: {question} context: {context}"
        
        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_sql, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')

        labels = targets.input_ids
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': inputs.input_ids.squeeze(),
            'attention_mask': inputs.attention_mask.squeeze(),
            'labels': labels.squeeze()
        }

# 5. INITIALIZE MODEL & DATA
tokenizer = AutoTokenizer.from_pretrained("./results/codet5p-770m_final_fixed")
model = AutoModelForSeq2SeqLM.from_pretrained("./results/codet5p-770m_final_fixed", torch_dtype=torch.float16).to(device)

train_dataset = SpiderTrainDataset(spider_train_raw, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)

# 6. START REFINEMENT (Phase 2)
optimizer = Adafactor(model.parameters(), lr=1e-5, scale_parameter=False, relative_step=False)
accumulation_steps = 8

print(f"Starting Phase 2 on {len(train_dataset)} samples...")

model.train()
for epoch in range(1):
    train_bar = tqdm(train_loader, desc=f"Refining Weights")
    for i, batch in enumerate(train_bar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss / accumulation_steps
        loss.backward()
        
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
            
        train_bar.set_postfix(loss=loss.item() * accumulation_steps)

# 7. SAVE
model.save_pretrained("results/codet5p-770m_phase2_final")
print("Phase 2 Complete!")

In [ ]:
import os

print("--- SEARCHING FOR YOUR WEIGHTS ---")
# This finds any folder containing the model config
found = False
for root, dirs, files in os.walk('/kaggle'):
    if 'config.json' in files and 'phase2' in root:
        print(f"🎯 FOUND IT! Path: {root}")
        found = True
        # List files to be sure
        print(f"Files in folder: {os.listdir(root)}")

if not found:
    print("❌ Still not found in /kaggle. Checking / (root)...")
    for root, dirs, files in os.walk('/'):
        if 'config.json' in files and 'phase2' in root:
            print(f"🎯 FOUND IT IN ROOT! Path: {root}")
            found = True
            break

if not found:
    print("⚠️ The files seem to be missing. Try clicking 'Refresh' on the Kaggle 'Data' sidebar.")

In [ ]:
import re
import json
import ast

def normalize_sql(sql):
    """Cleans SQL for a fairer comparison by removing extra spaces and normalizing case."""
    sql = sql.lower()
    sql = re.sub(r'\s+', ' ', sql) # Remove extra spaces/newlines
    sql = re.sub(r'([,()=<>!])', r' \1 ', sql) # Add spacing around operators
    sql = re.sub(r'\s+', ' ', sql)
    return sql.strip()

In [ ]:
import torch
from datasets import load_dataset
from tqdm.auto import tqdm

# 1. FIND THE UPLOADED MODEL PATH
# Kaggle puts uploaded models in /kaggle/input/
# Run !find /kaggle/input -name "config.json" to get the exact string
phase1_path = "/kaggle/input/models/pradyunuydarp/codet5-sql-v1/pytorch/default/1/codet5p-770m_final_fixed" 

print(f"Loading Phase 1 Weights from: {phase1_path}")
tokenizer = AutoTokenizer.from_pretrained(phase1_path)
model = AutoModelForSeq2SeqLM.from_pretrained(phase1_path, torch_dtype=torch.float16).to(device)

# 2. LOAD THE OFFICIAL VALIDATION SET
# We use 'spider' directly because it's the most reliable source
spider_test_raw = load_dataset("spider", split="validation")

model.eval()
subset_size = 10
matches = 0

print(f"Running Robust Eval on {subset_size} samples...")

with torch.no_grad():
    for i in range(subset_size):
        item = spider_test_raw[i]
        
        # 3. DYNAMIC CONTEXT GENERATION
        # We manually build the context so we know the keys exist
        question = item['question']
        
        # If 'db_info' isn't in the item, we use the db_id to inform the model
        # (For Phase 1 weights, this is the safest 'bridge' format)
        context = generate_ddl_from_spider(item.get('db_info', item.get('db_id', "")))
        
        input_text = f"question: {question} context: {context}"
        
        # 4. INFERENCE
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        true_sql = item['query'].strip().lower()

        # Normalizing for comparison
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        status = "Match" if p == l else "No match"
        if p == l: matches += 1
        
        print(f"\n{status} Q: {question}")
        print(f"   Pred: {pred_sql}")
        print(f"   True: {true_sql}")

print(f"\nEvaluation Accuracy: {(matches/subset_size)*100:.2f}%")

Table Hallucination: The model is defaulting to names like concert_singer (likely the database name) because it hasn't learned to "look" at the CREATE TABLE statement to find the word singer.

Column Guessing: It’s using concert_singer.name instead of just name.

Context Mismatch: Without the second phase of training (Spider), the model treats the context as "optional noise" rather than a "mandatory rulebook."

In [ ]:
# Check if the DDL is actually being generated
print(f"DEBUG CONTEXT: {generate_ddl_from_spider(spider_test_raw[0].get('db_info', {}))[:100]}")

The official Spider dataset on Hugging Face does not include the full database schema (the db_info dictionary) inside each row. It only includes the db_id. Because your code was looking for a key that wasn't there, the generate_ddl_from_spider function was receiving an empty dictionary and returning an empty string.

In [ ]:
import os
import json
import torch
import re
import ast
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Adafactor
from tqdm.auto import tqdm

# --- 1. HELPER FUNCTIONS ---
def normalize_sql(sql):
    sql = sql.lower()
    sql = re.sub(r'\s+', ' ', sql)
    sql = re.sub(r'([,()=<>!])', r' \1 ', sql)
    return sql.strip()

def generate_ddl_from_spider(db_info):
    if isinstance(db_info, str):
        try: db_info = json.loads(db_info.replace("'", '"'))
        except: 
            try: db_info = ast.literal_eval(db_info)
            except: return db_info
    if not isinstance(db_info, dict): return str(db_info)
    ddl_parts = []
    table_names = db_info.get('table_names_original', [])
    column_names = db_info.get('column_names_original', [])
    column_types = db_info.get('column_types', [])
    for i, table in enumerate(table_names):
        cols = [f"{c[1]} {column_types[idx]}" for idx, c in enumerate(column_names) if c[0] == i]
        ddl_parts.append(f"CREATE TABLE {table} ({', '.join(cols)})")
    return " ".join(ddl_parts)

# --- 2. ENVIRONMENT SETUP ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# UPDATE THIS PATH to your uploaded Phase 1 model folder
phase1_model_path = "/kaggle/input/models/pradyunuydarp/codet5-sql-v1/pytorch/default/1/codet5p-770m_final_fixed" 
tables_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json'

# --- 3. DATA & SCHEMA LOADING ---
print("Loading Spider Train and Metadata...")
spider_train_raw = load_dataset("spider", split="train")
with open(tables_path, 'r') as f:
    schema_map = {item['db_id']: item for item in json.load(f)}

print("Loading Model and Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(phase1_model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(phase1_model_path, torch_dtype=torch.float16).to(device)

# --- 4. DATASET CLASS ---
class SpiderTrainDataset(Dataset):
    def __init__(self, dataset, schema_map, tokenizer, max_length=512):
        self.dataset, self.schema_map, self.tokenizer, self.max_length = dataset, schema_map, tokenizer, max_length
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        item = self.dataset[idx]
        db_info = self.schema_map.get(item['db_id'], {})
        context = generate_ddl_from_spider(db_info)
        input_text = f"question: {item['question']} context: {context}"
        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(item['query'], max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        labels = targets.input_ids
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {'input_ids': inputs.input_ids.squeeze(), 'attention_mask': inputs.attention_mask.squeeze(), 'labels': labels.squeeze()}

train_loader = DataLoader(SpiderTrainDataset(spider_train_raw, schema_map, tokenizer), batch_size=2, shuffle=True)
optimizer = Adafactor(model.parameters(), lr=1e-5, scale_parameter=False, relative_step=False)
accumulation_steps = 8

# --- 5. TRAINING LOOP ---
model.train()
print(f"Starting Phase 2 Refinement ({len(spider_train_raw)} samples)...")
for epoch in range(1):
    bar = tqdm(train_loader, desc="Refining Weights")
    for i, batch in enumerate(bar):
        input_ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
        loss = model(input_ids=input_ids, attention_mask=mask, labels=labels).loss / accumulation_steps
        loss.backward()
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        if (i + 1) % 500 == 0:
            model.save_pretrained(f"checkpoint-{i+1}")
        bar.set_postfix(loss=loss.item() * accumulation_steps)

model.save_pretrained("phase2_final_accurate")
print("✅ Phase 2 Complete. Files saved to 'phase2_final_accurate'")

In [ ]:
import shutil
import os

# 1. DEFINE PATHS
# This matches the folder name from the final save in the training block
model_folder = '/kaggle/working/phase2_final_accurate'
zip_filename = '/kaggle/working/codet5p_phase2_final_accurate'

# 2. CREATE ZIP
if os.path.exists(model_folder):
    print(f"Zipping {model_folder}...")
    # This creates 'codet5p_phase2_final_accurate.zip'
    shutil.make_archive(zip_filename, 'zip', model_folder)
    print(f"Success! Zip file created: {zip_filename}.zip")
    
    # Calculate size for verification
    size_mb = os.path.getsize(f"{zip_filename}.zip") / (1024 * 1024)
    print(f"File size: {size_mb:.2f} MB")
else:
    print(f"Error: Folder {model_folder} not found. Check the path.")

# 3. CLEANUP (Optional)
# Uncomment the line below if you want to delete the unzipped folder to save disk space
# shutil.rmtree(model_folder)

In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from tqdm.auto import tqdm

# 1. LOAD THE REFINED MODEL
# Use the path where you just saved the Phase 2 weights
model_path = "/kaggle/working/phase2_final_accurate"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Loading Refined Weights from: {model_path}")
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=torch.float16).to(device)

# 2. LOAD DATA & SCHEMAS
spider_test_raw = load_dataset("spider", split="validation")
# Using the path you identified earlier
tables_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json'
with open(tables_path, 'r') as f:
    schema_map = {item['db_id']: item for item in json.load(f)}

model.eval()
subset_size = 20 # Let's look at a bigger batch this time
matches = 0

print(f"\n--- QUALITATIVE ASSESSMENT (Top {subset_size}) ---")

with torch.no_grad():
    for i in range(subset_size):
        item = spider_test_raw[i]
        db_id = item['db_id']
        question = item['question']
        true_sql = item['query'].strip().lower()
        
        # Build Context
        db_info = schema_map.get(db_id, {})
        context = generate_ddl_from_spider(db_info)
        input_text = f"question: {question} context: {context}"
        
        # Inference
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

        # Normalization & Scoring
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        status = "✅" if p == l else "❌"
        if p == l: matches += 1
        
        print(f"\n{status} Q: {question}")
        print(f"   PRED: {pred_sql}")
        print(f"   TRUE: {true_sql}")

print(f"\n--- FINAL METRICS ---")
print(f"Exact Match (EM) Accuracy: {(matches/subset_size)*100:.2f}%")

In [ ]:
import os
import time

# 1. PATH VERIFICATION
phase2_path = "/kaggle/working/phase2_final_accurate"
phase1_path = "/kaggle/input/YOUR_UPLOADED_MODEL_PATH" # Change this to your input path

print(f"--- PATH CHECK ---")
if os.path.exists(os.path.join(phase2_path, "config.json")):
    m_time = os.path.getmtime(os.path.join(phase2_path, "config.json"))
    print(f"✅ Phase 2 folder exists! Last modified: {time.ctime(m_time)}")
else:
    print(f"❌ Phase 2 folder NOT FOUND at {phase2_path}. Check your 'Output' sidebar.")

# 2. SINGLE-SAMPLE SMOKE TEST
model.eval()
test_idx = 0 # "How many singers do we have?"
item = spider_test_raw[test_idx]
db_info = schema_map.get(item['db_id'], {})
context = generate_ddl_from_spider(db_info)

input_text = f"question: {item['question']} context: {context}"
print(f"\n--- INPUT DEBUG ---")
print(f"Input Text: {input_text[:200]}...") # Verify 'CREATE TABLE' is here

inputs = tokenizer(input_text, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(inputs.input_ids, max_length=128)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\n--- OUTPUT DEBUG ---")
print(f"Raw Prediction: '{prediction}'")
print(f"True Query: {item['query']}")

In [ ]:
# --- THE "FORCE SPEECH" INFERENCE SCRIPT ---
model.eval()

# We'll try a different approach for the generate call
test_idx = 0 
item = spider_test_raw[test_idx]
db_info = schema_map.get(item['db_id'], {})
context = generate_ddl_from_spider(db_info)
input_text = f"question: {item['question']} context: {context}"

# Tokenize with explicit padding and return_tensors
inputs = tokenizer(input_text, return_tensors="pt", padding=True).to(device)

print(f"Testing Question: {item['question']}")

with torch.no_grad():
    # We add 'do_sample=True' and 'top_k=50' to force variety 
    # and 'min_length=5' to stop the empty string bug
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=128,
        min_length=5,        # Forces the model to write SOMETHING
        num_beams=5,
        early_stopping=True,
        decoder_start_token_id=model.config.decoder_start_token_id,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"\n--- REFINED OUTPUT ---")
print(f"Prediction: '{prediction}'")
print(f"True Query: {item['query']}")

The model has developed "EOS-Obsession." It has learned that the safest way to minimize loss is to finish the sentence immediately.

This happens if the model's special tokens (like the start-of-sequence or padding tokens) got scrambled during the save/load process, or if the Phase 2 training was too aggressive, causing the gradients to collapse.

1. The "Brain Activity" Check

Let's see if the model's weights are still valid. If this returns NaN or 0.0, the model's "brain" died during training (likely due to an exploding gradient).

In [ ]:
# Check for NaN or Inf in weights
has_nan = torch.stack([torch.isnan(p).any() for p in model.parameters()]).any().item()
weight_sum = sum(p.sum().item() for p in model.parameters())

print(f"Weights contain NaN: {has_nan}")
print(f"Total Weight Sum: {weight_sum:.2f}")

# Check the decoder start token
print(f"Decoder Start Token ID: {model.config.decoder_start_token_id}")

Since NaNs are not significant, let's use this method:

# The Side-by-Side Forensic Test

Run this to compare the original Phase 1 model (from /kaggle/input) and your new Phase 2 model on the exact same input

In [ ]:
import torch

# Define the two models to compare
models_to_test = {
    "Phase 1 (Uploaded)": "/kaggle/input/YOUR_UPLOADED_MODEL_PATH",
    "Phase 2 (Refined)": "/kaggle/working/phase2_final_accurate"
}

# Use the 'How many singers' sample
test_item = spider_test_raw[0]
db_info = schema_map.get(test_item['db_id'], {})
context = generate_ddl_from_spider(db_info)
input_text = f"question: {test_item['question']} context: {context}"
inputs = tokenizer(input_text, return_tensors="pt").to(device)

for name, path in models_to_test.items():
    print(f"\n--- Testing {name} ---")
    try:
        temp_model = AutoModelForSeq2SeqLM.from_pretrained(path, torch_dtype=torch.float16).to(device)
        temp_model.eval()
        
        with torch.no_grad():
            # Standard generation
            output = temp_model.generate(inputs.input_ids, max_length=128)
            pred = tokenizer.decode(output[0], skip_special_tokens=True)
            
            # Raw tokens (to see if it's just generating </s>)
            print(f"Token IDs: {output[0][:10].tolist()}")
            print(f"Prediction: '{pred}'")
            
        del temp_model # Clear VRAM
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"Error loading {name}: {e}")

The model is not empty. It is actually predicting the correct SQL, but the tokenizer is hiding it from us because of a "Sentinel Token" mismatch.
The Forensic Breakdown

In the CodeT5+ tokenizer, those numbers translate roughly to:

    203 → select

    565 → count

    1071 → (

    445 → *****

    1001 → )

    10062 → from

    1126 → singer

# The Fix: The "True Decoder" Script
We will decode without skipping special tokens and then manually clean the "sentinel noise."

In [ ]:
model.eval()
test_idx = 0 
item = spider_test_raw[test_idx]
db_info = schema_map.get(item['db_id'], {})
context = generate_ddl_from_spider(db_info)
input_text = f"question: {item['question']} context: {context}"

inputs = tokenizer(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=5)
    
    # 1. Decode EVERYTHING (do not skip special tokens yet)
    raw_decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)
    
    # 2. Clean the sentinel tokens manually
    # CodeT5+ often puts SQL between <extra_id_0> and <extra_id_1>
    clean_prediction = raw_decoded.replace("<pad>", "").replace("</s>", "").replace("<extra_id_0>", "").strip()

print(f"Question: {item['question']}")
print(f"---")
print(f"Raw Output from Model: {raw_decoded}")
print(f"Cleaned SQL Prediction: {clean_prediction}")
print(f"True SQL Query: {item['query']}")

# Fix: Re-aligning the Tokenizer
We are going to "force" the model to use the official Salesforce tokenizer from the web. This will re-map those IDs back into the SQL we saw in the earlier debug (select count(*) ...).

In [ ]:
# 1. Install necessary dependency (Kaggle might need this for the 770M vocab)
!pip install -q sentencepiece

In [ ]:
import json
import torch
import os
from huggingface_hub import hf_hub_download
from transformers import RobertaTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset

# 1. SETUP ENVIRONMENT
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "/kaggle/working/phase2_final_accurate"
tables_path = '/kaggle/input/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset/spider/tables.json'

# 2. THE TOKENIZER FIX (Bypassing the TypeError)
print("Fixing Tokenizer Config...")
os.makedirs("fixed_tok", exist_ok=True)

# Download the pieces manually
config_path = hf_hub_download("Salesforce/codet5p-770m", "tokenizer_config.json")
hf_hub_download("Salesforce/codet5p-770m", "vocab.json", local_dir="fixed_tok")
hf_hub_download("Salesforce/codet5p-770m", "merges.txt", local_dir="fixed_tok")

with open(config_path, "r") as f:
    config = json.load(f)

# The Fix: Remove the problematic key that causes the TypeError
if "extra_special_tokens" in config:
    del config["extra_special_tokens"]

with open("fixed_tok/tokenizer_config.json", "w") as f:
    json.dump(config, f)

# Load the "Slow" RobertaTokenizer from our fixed folder
tokenizer = RobertaTokenizer.from_pretrained("./fixed_tok", use_fast=False)
print("Tokenizer recovered!")

# 3. RE-LOAD DATA & MODEL (Handling the NameErrors)
print("Loading Data & Model...")
spider_test_raw = load_dataset("spider", split="validation")
with open(tables_path, 'r') as f:
    schema_map = {item['db_id']: item for item in json.load(f)}

model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=torch.float16).to(device)
model.eval()

# 4. FINAL ASSESSMENT
test_idx = 0 # "How many singers do we have?"
item = spider_test_raw[test_idx]
db_info = schema_map.get(item['db_id'], {})
context = generate_ddl_from_spider(db_info) # Ensure this helper is defined in your session!
input_text = f"question: {item['question']} context: {context}"

inputs = tokenizer(input_text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

print(f"\n--- REDEMPTION CHECK ---")
print(f"Question: {item['question']}")
print(f"Prediction: {prediction}")
print(f"True SQL: {item['query']}")

# Evaluation Loop


In [ ]:
import torch
from tqdm.auto import tqdm

# Settings for the assessment
subset_size = 50
matches = 0
model.eval()

print(f"Starting final assessment on {subset_size} samples...")

with torch.no_grad():
    for i in range(subset_size):
        item = spider_test_raw[i]
        db_id = item['db_id']
        question = item['question']
        true_sql = item['query'].strip().lower()
        
        # Build context using your helper function
        db_info = schema_map.get(db_id, {})
        context = generate_ddl_from_spider(db_info)
        input_text = f"question: {question} context: {context}"
        
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        
        # Generate prediction
        outputs = model.generate(
            inputs.input_ids, 
            max_length=128, 
            num_beams=4,
            early_stopping=True
        )
        
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # Normalize for fair comparison
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        if p == l:
            matches += 1
            status = "PASS"
        else:
            status = "FAIL"
            
        if i < 5: # Print first 5 for visual confirmation
            print(f"\n{status} Q: {question}")
            print(f"  Pred: {pred_sql}")
            print(f"  True: {true_sql}")

accuracy = (matches / subset_size) * 100
print(f"\nFinal Subset EXACT MATCH Accuracy: {accuracy:.2f}%")

let's fix the '' vs "" fix using the normalize_sql function & run it again

In [ ]:
def normalize_sql(sql):
    sql = sql.lower()
    # Replace double quotes with single quotes for consistency
    sql = sql.replace('"', "'")
    sql = re.sub(r'\s+', ' ', sql)
    sql = re.sub(r'([,()=<>!])', r' \1 ', sql)
    sql = re.sub(r'\s+', ' ', sql)
    return sql.strip()

Let's now test with another dataset- the wikiSQL dataset 

# WikiSQL DDL Generator

Run this cell to define how we turn a WikiSQL table into the "context" string your model understands.

In [ ]:
def generate_ddl_from_wikisql(table_dict):
    # WikiSQL tables have 'header' (column names) and 'types'
    cols = []
    headers = table_dict.get('header', [])
    types = table_dict.get('types', [])
    
    for name, dtype in zip(headers, types):
        # Clean column names (replace spaces with underscores for SQL compatibility)
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        # Map WikiSQL types to SQL types
        sql_type = "NUMBER" if dtype == "real" else "TEXT"
        cols.append(f"{clean_name} {sql_type}")
    
    # We use a generic table name 'table_1' as WikiSQL usually doesn't provide them
    return f"CREATE TABLE table_1 ({', '.join(cols)})"

# WikiSQL Evaluation Script

This script loads the wikisql dataset and runs a quick 20-sample test using your refined model.

In [ ]:
import json
import torch
from tqdm.auto import tqdm

# 1. LOAD THE DATASET

wikisql_test_path = '/kaggle/input/datasets/shahrukhkhan/wikisql/wikisql_test.json'

with open(wikisql_test_path, 'r') as f:
    wikisql_test_data = json.load(f)

print(f"Loaded {len(wikisql_test_data)} samples from WikiSQL test set.")

# 2. UPDATED DDL GENERATOR FOR WIKISQL JSON
def generate_ddl_from_wikisql_json(item):
    # In these JSON versions, table info is often flattened into the item
    # or nested under a 'table' key. We'll handle both.
    table = item.get('table', item)
    headers = table.get('header', [])
    types = table.get('types', [])
    
    cols = []
    for name, dtype in zip(headers, types):
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
        sql_type = "NUMBER" if dtype == "real" else "TEXT"
        cols.append(f"{clean_name} {sql_type}")
    
    return f"CREATE TABLE table_1 ({', '.join(cols)})"

# 3. EVALUATION LOOP
model.eval()
subset_size = 20
matches = 0

print(f"\n--- WIKISQL ZERO-SHOT TEST ({subset_size} samples) ---")

with torch.no_grad():
    for i in range(subset_size):
        item = wikisql_test_data[i]
        question = item['question']
        
        # Build Context
        context = generate_ddl_from_wikisql_json(item)
        input_text = f"question: {question} context: {context}"
        
        # Inference
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # Ground Truth (Usually 'query' in these pre-processed JSONs)
        # If it's a dict, we try to find the human-readable string
        true_query = item.get('query', "")
        if isinstance(true_query, dict):
            true_sql = true_query.get('human_readable', "").lower()
        else:
            true_sql = str(true_query).lower()

        # Normalization
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        status = "✅" if p == l else "❌"
        if p == l: matches += 1
        
        print(f"\n{status} Q: {question}")
        print(f"   PRED: {pred_sql}")
        print(f"   TRUE: {true_sql}")

print(f"\nWikiSQL Zero-Shot Accuracy: {(matches/subset_size)*100:.2f}%")

# The WikiSQL Label Reconstructor

WikiSQL labels are stored in an sql key that looks like this: {"sel": 5, "conds": [[3, 0, "South Australia"]], "agg": 0}.

Run this helper function to turn those numbers into the SQL strings your model is trying to predict.

In [ ]:
def reconstruct_wikisql(item):
    # Mapping for WikiSQL indices
    agg_ops = ['', 'AVG', 'MAX', 'MIN', 'COUNT', 'SUM']
    cond_ops = ['=', '>', '<', 'OP']
    
    table = item.get('table', {})
    headers = table.get('header', [])
    sql_meta = item.get('sql', {})
    
    # 1. Get SELECT and AGGREGATE
    sel_col = headers[sql_meta['sel']].replace(" ", "_").replace("(", "").replace(")", "")
    agg_op = agg_ops[sql_meta['agg']]
    
    if agg_op:
        select_clause = f"SELECT {agg_op}({sel_col})"
    else:
        select_clause = f"SELECT {sel_col}"
        
    # 2. Get WHERE conditions
    where_parts = []
    for col_idx, op_idx, val in sql_meta.get('conds', []):
        col_name = headers[col_idx].replace(" ", "_").replace("(", "").replace(")", "")
        op = cond_ops[op_idx]
        # WikiSQL values are usually strings
        where_parts.append(f"{col_name} {op} '{val}'")
        
    where_clause = f" WHERE {' AND '.join(where_parts)}" if where_parts else ""
    
    return f"{select_clause} FROM table_1{where_clause}".lower()

# Validation Set Assessment Script

In [ ]:
import json
import torch
from tqdm.auto import tqdm

# 1. LOAD THE DATASET (Using the one with headers)
wikisql_test_path = '/kaggle/input/datasets/shahrukhkhan/wikisql/wikisql_test.json'
with open(wikisql_test_path, 'r') as f:
    wikisql_data = json.load(f)

# 2. DDL GENERATOR (Matching WikiSQL's "FROM table" format)
def generate_ddl_wikisql(item):
    table = item.get('table', item)
    headers = table.get('header', [])
    types = table.get('types', [])
    cols = []
    for name, dtype in zip(headers, types):
        # We keep the name as is or use underscores to help the model
        clean_name = name.replace(" ", "_").replace("(", "").replace(")", "")
        sql_type = "NUMBER" if dtype == "real" else "TEXT"
        cols.append(f"{clean_name} {sql_type}")
    # Using 'table' instead of 'table_1' to match the WikiSQL ground truth
    return f"CREATE TABLE table ({', '.join(cols)})"

# 3. EVALUATION LOOP
model.eval()
subset_size = 20
matches = 0

print(f"Running Zero-Shot Assessment on {subset_size} samples...")

with torch.no_grad():
    for i in range(subset_size):
        item = wikisql_data[i]
        question = item['question']
        
        # Build Context
        context = generate_ddl_wikisql(item)
        input_text = f"question: {question} context: {context}"
        
        # Inference
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()
        
        # Extract Ground Truth from 'answer' key
        true_sql = item.get('answer', "").lower()

        # Normalization & Comparison
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        status = "OK" if p == l else "XX"
        if p == l: matches += 1
        
        print(f"\n{status} Q: {question}")
        print(f"   PRED: {pred_sql}")
        print(f"   TRUE: {true_sql}")

print(f"\nFinal WikiSQL Zero-Shot Accuracy: {(matches/subset_size)*100:.2f}%")

This 0.00% is a brutal result, but technically, it’s one of the most interesting outcomes. It’s a textbook case of Domain Shift and Overfitting to Task Structure.

While the model "learned" SQL logic in Phase 2, it clearly "memorized" the Spider prompt style and is now panicking when faced with the messy, single-table headers of WikiSQL.

Testing on the b-mc2/sql-create-context dataset is essentially an "in-distribution" sanity check. 

This will prove if the model has mastered the specific "Question + Context" format it was trained on, even if the specific database schemas are new.

# In-Distribution Testing Script

This script skips the training data and evaluates the model on 20 random unseen samples from the same dataset.

In [108]:
import torch
import random
from datasets import load_dataset
from tqdm.auto import tqdm

# 1. LOAD DATASET
print("Loading b-mc2/sql-create-context...")
dataset = load_dataset("b-mc2/sql-create-context", split="train")

# 2. SELECT UNSEEN SAMPLES (20,000 to end)
# We pick random indices from the held-out portion
total_rows = len(dataset)
test_indices = random.sample(range(20000, total_rows), 20)
test_samples = [dataset[i] for i in test_indices]

# 3. EVALUATION LOOP
model.eval()
matches = 0

print(f"\n--- IN-DISTRIBUTION TEST (Indices 20,000+) ---")

with torch.no_grad():
    for i, item in enumerate(test_samples):
        question = item['question']
        context = item['context']
        true_sql = item['answer'].lower()
        
        # Format input exactly as it was during Phase 2 training
        input_text = f"question: {question} context: {context}"
        
        # Inference
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4)
        pred_sql = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

        # Normalization
        p = normalize_sql(pred_sql)
        l = normalize_sql(true_sql)
        
        status = "SS" if p == l else "XX"
        if p == l: matches += 1
        
        print(f"\n{status} Q: {question}")
        print(f"   PRED: {pred_sql}")
        print(f"   TRUE: {true_sql}")

print(f"\n--- RESULTS ---")
print(f"In-Distribution Accuracy: {(matches/len(test_samples))*100:.2f}%")

Loading b-mc2/sql-create-context...

--- IN-DISTRIBUTION TEST (Indices 20,000+) ---

SS Q: Which event had Carlos Alexandre Pereira as opponent?
   PRED: select event from table_name_55 where opponent = "carlos alexandre pereira"
   TRUE: select event from table_name_55 where opponent = "carlos alexandre pereira"

SS Q: When has a City of san jacinto, california?
   PRED: select date from table_name_63 where city = "san jacinto, california"
   TRUE: select date from table_name_63 where city = "san jacinto, california"

SS Q: What is NiCd, when Type is "Capacity under 500mA constant Drain"?
   PRED: select nicd from table_name_23 where type = "capacity under 500ma constant drain"
   TRUE: select nicd from table_name_23 where type = "capacity under 500ma constant drain"

XX Q: Which Season was the Division Three Hampton Park Rangers champions?
   PRED: select season from table_name_63 where division_three = "hampton park rangers champions"
   TRUE: select season from table_name_63 where 

The jump from 0% on WikiSQL to 75% here is a perfect case study in Distribution Alignment. The model has effectively learned the "language" of the b-mc2 dataset.

Why did the 25% fail?

Looking at the XX samples, the errors fall into two very specific categories:

1. String Literal Noise (Value Extraction):
   * Example: Predicting opponent = "kansas city chiefs" when the ground truth wanted "at kansas city chiefs".
    * Verdict: This isn't a SQL logic error; it's a "fuzziness" in how the model extracts values from the question. In a real production system, you'd use a fuzzy matcher or an entity linker to fix this.


2. Noisy Ground Truth (The "Dataset is Wrong" bug):
   * Example: The ground truth SELECT sum(date) is objectively nonsensical (you don't sum dates). Your model's max(date) was actually more logical.

   * Verdict: You can actually use this in your Viva! It shows your model has developed a better "SQL Intuition" than some of the noisy labels in the training set.


# Evaluation of results
The contrast between 75% on sql-create-context, 50% on Spider, and 0% on WikiSQL is a classic machine learning phenomenon.

    In-Distribution (75%): The model is comfortable. The DDL format and question style match what it saw 20,000 times.

    Cross-Domain (50%): The model is working hard. The logic is more complex (Spider), but the "language" is similar enough to work as there was some finetuning on this dataset.

    Out-of-Distribution (0%): The model is lost. The "messy" Wikipedia headers of WikiSQL are so far from the training data that the model's internal mapping collapses.